# Lecture 4 · From one neuron to a dense layer and a batch

A dense layer is not a new kind of operation: **each output row is one familiar neuron**. We will first follow one row, stack two rows, send a vector gradient backward, and only then let three examples share the same `W` and `b`.

**Learning path:** one row → two-output dense layer → one backward pass → three-example batch → microbatch equivalence.

In [1]:
import html
import torch
from IPython.display import HTML, display

torch.set_default_dtype(torch.float64)

def fmt_number(value):
    value = float(value)
    return f"{value:.0f}" if abs(value - round(value)) < 1e-12 else f"{value:.4g}"

def fmt(value):
    if isinstance(value, torch.Tensor):
        value = value.detach()
        if value.ndim == 0:
            return fmt_number(value.item())
        return '[' + ', '.join(fmt(v) for v in value) + ']'
    return str(value)

def show_table(headers, rows, caption=None):
    head = ''.join(f'<th>{html.escape(str(h))}</th>' for h in headers)
    body = ''.join(
        '<tr>' + ''.join(f'<td>{html.escape(fmt(v))}</td>' for v in row) + '</tr>'
        for row in rows
    )
    cap = f'<div class="cap">{html.escape(caption)}</div>' if caption else ''
    display(HTML(f'''
    <style>
      .dense-card {{max-width:920px;padding:14px 16px;border:1px solid #cbd5e1;border-radius:14px;background:#f8fafc;color:#17202a}}
      .dense-card table {{border-collapse:collapse;width:100%;font:14px/1.4 ui-monospace,SFMono-Regular,Menlo,monospace}}
      .dense-card th {{background:#183b4e;color:white;text-align:left}}
      .dense-card th,.dense-card td {{padding:8px 10px;border:1px solid #cbd5e1;vertical-align:top}}
      .dense-card tr:nth-child(even) td {{background:#eef6f8}}
      .dense-card .cap {{font:600 14px/1.3 system-ui;margin:0 0 9px;color:#183b4e}}
    </style><div class="dense-card">{cap}<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'''))

print(f"PyTorch {torch.__version__} · float64 · deterministic tensors · no downloads")

/private/tmp/l3-dense-exec-venv311/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch 2.13.0 · float64 · deterministic tensors · no downloads


## 1 · One output row is one neuron

For the first output, the first row of $W$ supplies the neuron's weights: $w_1=(1,3)$ and $b_1=0$. With $x=(2,-1)$, predict $z_1=w_1^T x+b_1$ before running the cell.

In [2]:
x = torch.tensor([2.0, -1.0])
w1 = torch.tensor([1.0, 3.0])
b1 = torch.tensor(0.0)
products = w1 * x
z1 = w1 @ x + b1

torch.testing.assert_close(products, torch.tensor([2.0, -3.0]))
torch.testing.assert_close(z1, torch.tensor(-1.0))

display(HTML(f'''
<div style="display:flex;flex-wrap:wrap;align-items:center;gap:10px;font:600 16px system-ui;color:#17202a">
  <span style="padding:10px 14px;border-radius:12px;background:#e8f2f6">$1\cdot2=2$</span>
  <span style="font-size:22px">+</span>
  <span style="padding:10px 14px;border-radius:12px;background:#e8f2f6">$3\cdot(-1)=-3$</span>
  <span style="font-size:22px">+</span>
  <span style="padding:10px 14px;border-radius:12px;background:#fff3cd">$b_1=0$</span>
  <span style="font-size:22px">→</span>
  <span style="padding:10px 14px;border-radius:12px;background:#d9f2e6;color:#0c6046">$z_1={z1.item():g}$</span>
</div>'''))
print('The first row computed one dot product. Nothing is batched yet.')

The first row computed one dot product. Nothing is batched yet.


## 2 · A dense layer stacks those neurons

The second row $(w_2,b_2)=((-2,1),1)$ computes a second output from the **same** input. Stacking the two row calculations gives $z=Wx+b$.

In [3]:
W = torch.tensor([[1.0, 3.0],
                  [-2.0, 1.0]])
b = torch.tensor([0.0, 1.0])
z = W @ x + b

torch.testing.assert_close(z, torch.tensor([-1.0, -4.0]))
show_table(
    ['output row', 'weights', 'bias', 'calculation', 'stored output'],
    [
        ['row 1', W[0], b[0], '1·2 + 3·(-1) + 0', z[0]],
        ['row 2', W[1], b[1], '-2·2 + 1·(-1) + 1', z[1]],
    ],
    'Two neurons run in parallel; stacking their outputs produces z.'
)
print('shape check:', tuple(W.shape), '@', tuple(x.shape), '+', tuple(b.shape), '→', tuple(z.shape))

output row,weights,bias,calculation,stored output
row 1,"[1, 3]",0,1·2 + 3·(-1) + 0,-1
row 2,"[-2, 1]",1,-2·2 + 1·(-1) + 1,-4


shape check: (2, 2) @ (2,) + (2,) → (2,)


## 3 · One vector arrives from the rest of the graph

Suppose the later loss sends $g_z=\partial L/\partial z=(4,-2)$ back to this layer. This is the vector version of the earlier scalar *upstream gradient*: $g_1=4$ arrives at output row 1 and $g_2=-2$ at row 2.

For output row $i$, write $z_i=\sum_j W_{ij}x_j+b_i$. Reuse the scalar affine rule coordinate by coordinate:

$$x_j\text{ receives }g_iW_{ij},\qquad W_{ij}\text{ receives }g_ix_j,\qquad b_i\text{ receives }g_i.$$

The same $x_j$ feeds every output row, so its returns **add**: $(g_x)_j=\sum_i g_iW_{ij}$. Separate parameter rows instead **stack**. Only now do we recognize the compact formulas

$$g_x=W^Tg_z,\qquad g_W=g_zx^T,\qquad g_b=g_z.$$

Generic shapes verify the orientation: $(d\times1)=(d\times m)(m\times1)$ and $(m\times d)=(m\times1)(1\times d)$. Shapes check the derivation; they do not replace it.

In [4]:
g_z = torch.tensor([4.0, -2.0])
gx_row_contrib = g_z[:, None] * W          # row i returns g_i w_i to shared x
gW_row_contrib = g_z[:, None] * x[None, :] # row i receives g_i x^T
g_x_manual = gx_row_contrib.sum(dim=0)
g_W_manual = gW_row_contrib
g_b_manual = g_z.clone()

torch.testing.assert_close(gx_row_contrib, torch.tensor([[4.0, 12.0], [4.0, -2.0]]))
torch.testing.assert_close(g_x_manual, torch.tensor([8.0, 10.0]))
torch.testing.assert_close(g_W_manual, torch.tensor([[8.0, -4.0], [-4.0, 2.0]]))
torch.testing.assert_close(g_b_manual, torch.tensor([4.0, -2.0]))

show_table(
    ['output row', 'arrival g_i', 'return to shared x', 'return to its W row'],
    [
        ['row 1', g_z[0], gx_row_contrib[0], gW_row_contrib[0]],
        ['row 2', g_z[1], gx_row_contrib[1], gW_row_contrib[1]],
    ],
    'Run the familiar scalar affine rule once per output row'
)
show_table(
    ['recipient', 'how row returns combine', 'stacked formula', 'generic operand shapes'],
    [
        ['x', 'add [4,12] + [4,-2]', 'Wᵀ g_z = [8,10]', '(d×m)(m×1) → d×1'],
        ['W', 'stack the two returned rows', 'g_z xᵀ', '(m×1)(1×d) → m×d'],
        ['b', 'one return per output bias', 'g_z', 'm×1'],
    ],
    'Coordinate rules derive the gradients; shapes verify their orientation.'
)

x_ag = x.clone().requires_grad_()
W_ag = W.clone().requires_grad_()
b_ag = b.clone().requires_grad_()
z_ag = W_ag @ x_ag + b_ag
z_ag.backward(g_z)

torch.testing.assert_close(z_ag, z)
torch.testing.assert_close(x_ag.grad, g_x_manual)
torch.testing.assert_close(W_ag.grad, g_W_manual)
torch.testing.assert_close(b_ag.grad, g_b_manual)
print('✓ PyTorch reproduces z, g_x, g_W, and g_b exactly.')

output row,arrival g_i,return to shared x,return to its W row
row 1,4,"[4, 12]","[8, -4]"
row 2,-2,"[4, -2]","[-4, 2]"


recipient,how row returns combine,stacked formula,generic operand shapes
x,"add [4,12] + [4,-2]","Wᵀ g_z = [8,10]",(d×m)(m×1) → d×1
W,stack the two returned rows,g_z xᵀ,(m×1)(1×d) → m×d
b,one return per output bias,g_z,m×1


✓ PyTorch reproduces z, g_x, g_W, and g_b exactly.


## 4 · A batch adds an example axis, not new parameters

An individual $x^{(n)}$ is still a $d\times1$ column. PyTorch stacks the transposes as rows $X\in\mathbb{R}^{B\times d}$, so the batch forward is

$$Z=XW^T+\mathbf{1}_B b^T,$$

which code writes as `X @ W.T + b`. Every row reuses the same $W,b$. For each example $n$, define

$$r^{(n)}=z^{(n)}-y^{(n)},\qquad \ell_n=\tfrac12\lVert r^{(n)}\rVert_2^2.$$

Coordinate calculus gives $\partial\ell_n/\partial z_k^{(n)}=z_k^{(n)}-y_k^{(n)}=r_k^{(n)}$. Because the scalar batch loss is the **mean** $L=\frac1B\sum_n\ell_n$, its arrival at the batch output is $G_Z=\partial L/\partial Z=R/B$. The targets below are deliberately constructed so row 1 has residual $(4,-2)$, matching the preceding single-example case.

In [5]:
X = torch.tensor([[ 2.0, -1.0],
                  [-1.0,  2.0],
                  [ 2.0,  2.0]])
Y = torch.tensor([[-5.0, -2.0],
                  [ 7.0,  1.0],
                  [ 7.0, -2.0]])
B = X.shape[0]
Z = X @ W.T + b
G = Z - Y                         # dℓ_i / dz_i
loss_each = 0.5 * (G**2).sum(dim=1)

expected_Z = torch.tensor([[-1.0, -4.0], [5.0, 5.0], [8.0, -1.0]])
expected_G = torch.tensor([[4.0, -2.0], [-2.0, 4.0], [1.0, 1.0]])
expected_losses = torch.tensor([10.0, 10.0, 1.0])
torch.testing.assert_close(Z, expected_Z)
torch.testing.assert_close(G, expected_G)
torch.testing.assert_close(loss_each, expected_losses)

show_table(
    ['i', 'x_i', 'z_i', 'y_i', 'g_i = z_i − y_i', 'ℓ_i'],
    [[i + 1, X[i], Z[i], Y[i], G[i], loss_each[i]] for i in range(B)],
    'Forward pass and the gradient emitted by each example’s loss'
)
print('mean batch loss L =', loss_each.mean().item())

i,x_i,z_i,y_i,g_i = z_i − y_i,ℓ_i
1,"[2, -1]","[-1, -4]","[-5, -2]","[4, -2]",10
2,"[-1, 2]","[5, 5]","[7, 1]","[-2, 4]",10
3,"[2, 2]","[8, -1]","[7, -2]","[1, 1]",1


mean batch loss L = 7.0


## 5 · Shared paths add; the mean reduction then scales

Before reduction, example $n$ proposes $r^{(n)}(x^{(n)})^T$ to `W`, $r^{(n)}` to `b`, and $W^Tr^{(n)}` to its own input row. The shared `W,b` receive the **sum** of all example paths. Our declared scalar loss is a **mean**, so that sum is divided by $B=3$. Distinct input examples do not add into one another; their gradients remain separate rows of `X.grad`.

In [6]:
gW_each = torch.stack([torch.outer(G[i], X[i]) for i in range(B)])
gb_each = G.clone()
gx_each_unreduced = G @ W

expected_gW_each = torch.tensor([
    [[ 8.0, -4.0], [-4.0, 2.0]],
    [[ 2.0, -4.0], [-4.0, 8.0]],
    [[ 2.0,  2.0], [ 2.0, 2.0]],
])
expected_gb_each = torch.tensor([[4.0, -2.0], [-2.0, 4.0], [1.0, 1.0]])
expected_gx_each = torch.tensor([[8.0, 10.0], [-10.0, -2.0], [-1.0, 4.0]])
torch.testing.assert_close(gW_each, expected_gW_each)
torch.testing.assert_close(gb_each, expected_gb_each)
torch.testing.assert_close(gx_each_unreduced, expected_gx_each)

show_table(
    ['i', 'g_i x_iᵀ → W', 'g_i → b', 'Wᵀg_i → x_i'],
    [[i + 1, gW_each[i], gb_each[i], gx_each_unreduced[i]] for i in range(B)],
    'Unreduced per-example contributions'
)

gW_sum, gb_sum = gW_each.sum(dim=0), gb_each.sum(dim=0)
gW_mean, gb_mean = gW_each.mean(dim=0), gb_each.mean(dim=0)
expected_gW_sum = torch.tensor([[12.0, -6.0], [-6.0, 12.0]])
expected_gb_sum = torch.tensor([3.0, 3.0])
torch.testing.assert_close(gW_sum, expected_gW_sum)
torch.testing.assert_close(gb_sum, expected_gb_sum)
torch.testing.assert_close(gW_mean, expected_gW_sum / B)
torch.testing.assert_close(gb_mean, expected_gb_sum / B)
show_table(
    ['reduction', 'gradient for W', 'gradient for b'],
    [
        ['sum of 3 contributions', gW_sum, gb_sum],
        ['mean used by L', gW_mean, gb_mean],
    ],
    'The loss reduction controls the scale, not the direction of accumulation.'
)

i,g_i x_iᵀ → W,g_i → b,Wᵀg_i → x_i
1,"[[8, -4], [-4, 2]]","[4, -2]","[8, 10]"
2,"[[2, -4], [-4, 8]]","[-2, 4]","[-10, -2]"
3,"[[2, 2], [2, 2]]","[1, 1]","[-1, 4]"


reduction,gradient for W,gradient for b
sum of 3 contributions,"[[12, -6], [-6, 12]]","[3, 3]"
mean used by L,"[[4, -2], [-2, 4]]","[1, 1]"


## 6 · Full-batch autograd reproduces the mean

We now rebuild the same forward pass with tracked tensors. `retain_grad()` lets us inspect the non-leaf matrix `Z`; ordinary training usually does not retain it.

In [7]:
X_full = X.clone().requires_grad_()
W_full = W.clone().requires_grad_()
b_full = b.clone().requires_grad_()
Z_full = X_full @ W_full.T + b_full
Z_full.retain_grad()
loss_full = 0.5 * ((Z_full - Y)**2).sum(dim=1).mean()
loss_full.backward()

torch.testing.assert_close(loss_full, torch.tensor(7.0))
torch.testing.assert_close(Z_full, expected_Z)
torch.testing.assert_close(Z_full.grad, expected_G / B)
torch.testing.assert_close(W_full.grad, gW_mean)
torch.testing.assert_close(b_full.grad, gb_mean)
torch.testing.assert_close(X_full.grad, expected_gx_each / B)

show_table(
    ['autograd buffer', 'value', 'manual quantity matched'],
    [
        ['Z.grad', Z_full.grad, 'G / 3'],
        ['W.grad', W_full.grad, 'Σ(g_i x_iᵀ) / 3'],
        ['b.grad', b_full.grad, 'Σg_i / 3'],
        ['X.grad', X_full.grad, 'stack(Wᵀg_i) / 3'],
    ],
    'One backward call on the mean batch loss'
)
print('✓ Full-batch PyTorch matches every manual forward and backward value.')

autograd buffer,value,manual quantity matched
Z.grad,"[[1.333, -0.6667], [-0.6667, 1.333], [0.3333, 0.3333]]",G / 3
W.grad,"[[4, -2], [-2, 4]]",Σ(g_i x_iᵀ) / 3
b.grad,"[1, 1]",Σg_i / 3
X.grad,"[[2.667, 3.333], [-3.333, -0.6667], [-0.3333, 1.333]]",stack(Wᵀg_i) / 3


✓ Full-batch PyTorch matches every manual forward and backward value.


## 7 · Three microbatches give the same gradient

A backward call **adds** into `.grad`. To match a mean over three examples, each one-example loss contributes $\ell_i/3$. We intentionally do not clear `W_micro.grad` or `b_micro.grad` until all three contributions have arrived.

In [8]:
W_micro = W.clone().requires_grad_()
b_micro = b.clone().requires_grad_()
snapshots = []

for i in range(B):
    z_i = W_micro @ X[i] + b_micro
    scaled_loss_i = 0.5 * ((z_i - Y[i])**2).sum() / B
    scaled_loss_i.backward()
    snapshots.append((i + 1, W_micro.grad.detach().clone(), b_micro.grad.detach().clone()))

expected_W_snapshots = torch.stack([
    expected_gW_each[0] / B,
    expected_gW_each[:2].sum(dim=0) / B,
    expected_gW_each.sum(dim=0) / B,
])
expected_b_snapshots = torch.stack([
    expected_gb_each[0] / B,
    expected_gb_each[:2].sum(dim=0) / B,
    expected_gb_each.sum(dim=0) / B,
])
for j, (_, w_snapshot, b_snapshot) in enumerate(snapshots):
    torch.testing.assert_close(w_snapshot, expected_W_snapshots[j])
    torch.testing.assert_close(b_snapshot, expected_b_snapshots[j])

torch.testing.assert_close(W_micro.grad, W_full.grad)
torch.testing.assert_close(b_micro.grad, b_full.grad)
show_table(
    ['after example', 'accumulated W.grad', 'accumulated b.grad'],
    snapshots,
    'Each scaled microbatch adds one path contribution to the shared parameters.'
)
print('✓ Three scaled microbatches exactly equal one full-batch mean backward pass.')

after example,accumulated W.grad,accumulated b.grad
1,"[[2.667, -1.333], [-1.333, 0.6667]]","[1.333, -0.6667]"
2,"[[3.333, -2.667], [-2.667, 3.333]]","[0.6667, 0.6667]"
3,"[[4, -2], [-2, 4]]","[1, 1]"


✓ Three scaled microbatches exactly equal one full-batch mean backward pass.


## Takeaway

- A dense layer stacks neurons: row $j$ of $W$ computes output $z_j$.
- One arriving vector gives row-level scalar returns; shared input returns add into $g_x=W^Tg_z$, while separate parameter rows stack into $g_W=g_zx^T$ and $g_b=g_z$.
- With a row-stacked batch, $G_Z=R/B$, $g_W=G_Z^TX$, $g_b=G_Z^T\mathbf{1}_B$, and $g_X=G_ZW$.
- Shared-parameter contributions add; distinct input rows remain separate; a mean loss divides all returns by the batch size.
- Microbatch accumulation matches full-batch backprop only when the microbatch losses use the same overall reduction.